In [37]:
import pandas as pd
from pathlib import Path
import re
from concurrent.futures import ThreadPoolExecutor


In [38]:
import os

print("Number of logical CPUs:", os.cpu_count())


Number of logical CPUs: 12


In [39]:
file_path = r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\X\dataset_tweet-scraper_2025-05-01_17-51-50-726.csv"

# columns
# 1. author/fastFollowersCount - מספר המעקבים המהירים של המחבר
# 2. author/followersCount - מספר המעקבים של המחבר
# 3. author/friendsCount - מספר החברים של המחבר
# 4. author/listedCount - מספר הרשימות של המחבר
# 5. author/mediaCount - מספר הפוסטים של המחבר
# 6. author/statusesCount - מספר הפוסטים של המחבר
# 7. author/verified - מציין אם המחבר מאומת
# 8. author/statusesCount - המספר הכולל של פוסטים/ציוצים שהמחבר פרסם
# 9. bookmarkCount - מספר הפעמים שהפוסט נשמר למועדפים
# 10. fullText - תוכן הטקסט של הפוסט
# 11. entities/media/0/additional_media_info/monetizable - האם המדיה ניתנת למונטיזציה
# 12. likeCount - מספר הלייקים של הפוסט
# 13. quote/replyCount - מספר הציוצים של הפוסט
# 14. quote/retweetCount - מספר השיתופים של הפוסט
# 15. replyCount - מספר התגובות של הפוסט
# 16. retweetCount - מספר השיתופים של הפוסט
# 17. quoteCount - מתייחסת למספר הפעמים שפוסט מסוים צוטט על ידי משתמשים אחרים
# 18. quotedTweet/fullText - תוכן הטקסט של הפוסט שהוצג בציוצ
# 19. replyCount - מספר התגובות של הפוסט
# 20. viewCount - מספר הצפיות של הפוסט
# 21. text - תוכן הטקסט של הפוסט

columns_names_df = pd.read_csv(file_path)
# print(columns_names_df.columns.tolist())

selected_columns = ["author/fastFollowersCount", "author/followers",
                     "author/mediaCount", "author/statusesCount",
                    "author/isVerified", "author/statusesCount", "bookmarkCount",
                    "fullText", "entities/media/0/additional_media_info/monetizable",
                    "likeCount", "quote/replyCount", "quote/retweetCount", "replyCount",
                    "retweetCount", "quoteCount", "fullText", "replyCount",
                    "viewCount", "text"]

selected_columns_df = columns_names_df.loc[:, selected_columns]

selected_columns_names = selected_columns_df.columns

selected_columns_names

selected_columns_names = selected_columns_df.columns.tolist()

C:\Users\itama\AppData\Local\Temp\ipykernel_16128\4218522879.py:26: DtypeWarning: Columns (1494) have mixed types. Specify dtype option on import or set low_memory=False.
  columns_names_df = pd.read_csv(file_path)


In [40]:
def find_csv_files(folder_root):
    main_folder_path = Path(folder_root)
    if not main_folder_path.exists() or not main_folder_path.is_dir():
        raise ValueError(f"{main_folder_path} not exists")
    
    print(f"{main_folder_path} patent dir")


    csv_files_per_folder = {}

    for dir in main_folder_path.iterdir():
        if dir.is_dir() and re.match(r"[A-Za-z_-]+", dir.name.strip()):
            print(f"Found directory: {dir.name}")
            csv_folder = dir/ "CSV"
            if csv_folder.is_dir():
                csv_files = list(csv_folder.glob("*.csv"))
                csv_files_per_folder[dir.name] = {
                    "files": csv_files,
                    "count": len(csv_files)
                }
                
    # print("csv files per folder:", csv_files_per_folder)
                
            for i, file in enumerate(csv_files):
                print(f"📁 {dir.name} → 📄 {file.name}")
                if i == len(csv_files) - 1:
                    print(f"📁 {dir.name} – סך הכל {len(csv_files)} קבצים\n")

                      
    return csv_files_per_folder 
csv_files_per_folder = find_csv_files("C:/Users/itama/OneDrive - Ariel University/Documents/Desktop/programing/Research/Data Leakage/Data/X")


C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\X patent dir
Found directory: charity
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_15-51-57-212.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_15-55-41-119.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_20-37-24-476.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_20-43-17-365.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_20-49-10-658.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_21-20-59-885.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_21-27-34-494.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_21-34-07-118.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_21-35-30-122.csv
📁 charity – סך הכל 9 קבצים

Found directory: hiring
📁 hiring → 📄 dataset_tweet-scraper_2025-05-01_17-10-20-783.csv
📁 hiring → 📄 dataset_tweet-scraper_2025-05-01_19-36-28-580.csv
📁 hiring → 📄 dataset_tweet-scraper_2025-05-01_19-42-33-856.csv
📁 hiring → 📄 dataset_tweet-scraper_2025-05-04_

In [41]:
def merge_csv_files(csv_files_per_folder, dedup_columns=None):
    all_data = []

    for _, data in csv_files_per_folder.items():
        for file in data["files"]:
            try:
                df = pd.read_csv(file, low_memory=False)
                existing_columns = [col for col in selected_columns_names if col in df.columns]
                # Remove duplicates while preserving order
                existing_columns = list(dict.fromkeys(existing_columns))
                df = df[existing_columns]
                df["subject"] = file.parent.parent.name  # שם התיקייה העליונה כ־subject
                all_data.append(df)
            except Exception as e:
                print(f"⚠️ Error reading {file.name}: {e}")

    if not all_data:
        print("�� No data loaded.")
        return None

    merged_df = pd.concat(all_data, ignore_index=True)

    before = len(merged_df)
    if dedup_columns:
        valid_dedup_cols = [col for col in dedup_columns if col in merged_df.columns]
        merged_df.drop_duplicates(subset=valid_dedup_cols, inplace=True)
        print(f"🧹 Removed {before - len(merged_df)} duplicates using: {valid_dedup_cols}")
    else:
        merged_df.drop_duplicates(inplace=True)
        print(f"🧹 Removed {before - len(merged_df)} total duplicates")

    output_path = r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\X\x dataset.csv"
    merged_df.to_csv(output_path, index=False)
    print(f"✅ Merged dataset saved to '{output_path}'")

    return merged_df

In [42]:
# def merge_csv_files(csv_files_per_folder, dedup_columns=None):
#     all_data = []

#     for _, data in csv_files_per_folder.items():
#         for file in data["files"]:
#             try:
#                 df = pd.read_csv(file, low_memory=False)
#                 existing_columns = [col for col in selected_columns_names if col in df.columns]
#                 df = df[existing_columns]
#                 df["subject"] = file.parent.parent.name  # שם התיקייה העליונה כ־subject
#                 all_data.append(df)
#             except Exception as e:
#                 print(f"⚠️ Error reading {file.name}: {e}")

#     if not all_data:
#         print("🚫 No data loaded.")
#         return None

#     merged_df = pd.concat(all_data, ignore_index=True)

#     before = len(merged_df)
#     if dedup_columns:
#         valid_dedup_cols = [col for col in dedup_columns if col in merged_df.columns]
#         merged_df.drop_duplicates(subset=valid_dedup_cols, inplace=True)
#         print(f"🧹 Removed {before - len(merged_df)} duplicates using: {valid_dedup_cols}")
#     else:
#         merged_df.drop_duplicates(inplace=True)
#         print(f"🧹 Removed {before - len(merged_df)} total duplicates")

#     output_path = r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\X\x dataset.csv"
#     merged_df.to_csv(output_path, index=False)
#     print(f"✅ Merged dataset saved to '{output_path}'")

#     return merged_df


In [43]:
# def merge_csv_files(csv_files_per_folder):

#     all_data = []

#     for _, data in csv_files_per_folder.items():
#         for file in data["files"]:
#             df = pd.read_csv(file)
#             existing_columns = [col for col in selected_columns_names if col in df.columns]
#             df = df[existing_columns]  # שומר רק את העמודות שקיימות
#             all_data.append(df)


#     for file in data["files"]:
#         file_columns = pd.read_csv(file, nrows=0).columns.tolist()

#     # מציאת עמודות חסרות
#     missing_columns = [col for col in selected_columns_names if col not in file_columns]

#     if missing_columns:
#         print(f"⚠️ Warning: Missing columns in {file.name}: {missing_columns}")

    
#     merged_df = pd.concat(all_data, ignore_index=True)
#     merged_df.to_csv(r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\X\x dataset.csv", index=False)
#     print("All CSV files have been merged into 'x dataset.csv'")
#     print("✅ All CSV files have been merged into 'x dataset.csv'")

In [44]:

def create_doc_row(file_path, subject):
    df = pd.read_csv(file_path)
    num_post = len(df)

    if "twitterUrl" in df.columns and not df["twitterUrl"].isna().all():
        fb_url = df["twitterUrl"].dropna().iloc[0]
    else:
        fb_url = ""

    parameters = ",".join(df.columns)

    return {
        "URL": fb_url,
        "Number of posts": num_post,
        "subject": subject,
        "parameters": parameters
    }


# def generate_documentation(csv_files_per_folder, output_path="x Scraping documentation.xlsx"):
#     task_list = [
#         (file, folder)
#         for folder, data in csv_files_per_folder.items()
#         for file in data["files"]
#     ]

#     def row_creator(task):
#         file_path, subject = task
#         return create_doc_row(file_path, subject)

#     with ThreadPoolExecutor(max_workers=8) as executor:
#         documentation_rows = list(executor.map(row_creator, task_list))

#     # output_path = r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\Facebook"
#     documentation_df = pd.DataFrame(documentation_rows)
#     documentation_df.to_excel(r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\X\x Scraping documentation.xlsx", index=False)
#     print(f"📄 Documentation saved to: {output_path}")


In [45]:
def generate_documentation(csv_files_per_folder, merged_df,
                           output_path=r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\X\x Scraping documentation.xlsx"):

    documentation_rows = []

    for subject, data in csv_files_per_folder.items():
        for file in data["files"]:
            try:
                file_df = pd.read_csv(file, low_memory=False)
                existing_columns = [col for col in selected_columns_names if col in file_df.columns]
                file_df = file_df[existing_columns]
                file_df["subject"] = subject

                merged_subset = merged_df[
                    (merged_df["subject"] == subject) &
                    (merged_df[existing_columns].apply(tuple, axis=1).isin(file_df[existing_columns].apply(tuple, axis=1)))
                ]

                num_posts = len(merged_subset)

                fb_url = ""
                if "author/twitterUrl" in merged_subset.columns:
                    unique_urls = merged_subset["author/twitterUrl"].dropna().unique()
                    if len(unique_urls) == 1:
                        fb_url = unique_urls[0]

                parameters = ", ".join(existing_columns)

                documentation_rows.append({
                    "URL": fb_url,
                    "Number of posts": num_posts,
                    "subject": subject,
                    "parameters": parameters
                })

            except Exception as e:
                print(f"⚠️ Error processing file {file.name}: {e}")

    documentation_df = pd.DataFrame(documentation_rows)

    blank_rows = pd.DataFrame([{}, {}, {}])
    summary_row = pd.DataFrame([{
        "URL": "",
        "Number of posts": documentation_df["Number of posts"].sum(),
        "subject": "TOTAL (after deduplication)",
        "parameters": ""
    }])

    final_df = pd.concat([documentation_df, blank_rows, summary_row], ignore_index=True)
    final_df.to_excel(output_path, index=False)

    print(f"📄 Full documentation saved to: {output_path}")


In [46]:
folder_root = r"C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\X"

csv_files_per_folder = find_csv_files(folder_root)

merged_df = merge_csv_files(csv_files_per_folder)

generate_documentation(csv_files_per_folder, merged_df)


C:\Users\itama\OneDrive - Ariel University\Documents\Desktop\programing\Research\Data Leakage\Data\X patent dir
Found directory: charity
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_15-51-57-212.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_15-55-41-119.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_20-37-24-476.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_20-43-17-365.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_20-49-10-658.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_21-20-59-885.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_21-27-34-494.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_21-34-07-118.csv
📁 charity → 📄 dataset_tweet-scraper_2025-05-04_21-35-30-122.csv
📁 charity – סך הכל 9 קבצים

Found directory: hiring
📁 hiring → 📄 dataset_tweet-scraper_2025-05-01_17-10-20-783.csv
📁 hiring → 📄 dataset_tweet-scraper_2025-05-01_19-36-28-580.csv
📁 hiring → 📄 dataset_tweet-scraper_2025-05-01_19-42-33-856.csv
📁 hiring → 📄 dataset_tweet-scraper_2025-05-04_